In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
from pathlib import Path

repo_path = Path("/content/gdrive/MyDrive/GeoPulse-GitHub-Repo")

print("Repository exists:", repo_path.exists())
print("Project folder:", repo_path)

Repository exists: True
Project folder: /content/gdrive/MyDrive/GeoPulse-GitHub-Repo


In [3]:
files_to_check = [
    "data/synthetic/realistic_urban_mobility.csv",
    "data/synthetic/store_locations.csv",
    "data/synthetic/store_catchments_500m.csv"
]

for file in files_to_check:
    path = repo_path / file
    print(f"{file}: {'FOUND' if path.exists() else 'NOT FOUND'}")

data/synthetic/realistic_urban_mobility.csv: FOUND
data/synthetic/store_locations.csv: FOUND
data/synthetic/store_catchments_500m.csv: FOUND


In [4]:
import pandas as pd

gps_path = repo_path / "data/synthetic/realistic_urban_mobility.csv"
stores_path = repo_path / "data/synthetic/store_locations.csv"
catchments_path = repo_path / "data/synthetic/store_catchments_500m.csv"

gps_pd = pd.read_csv(gps_path)
stores_pd = pd.read_csv(stores_path)
catchments_pd = pd.read_csv(catchments_path)

print("GPS dataset shape:", gps_pd.shape)
print("Store dataset shape:", stores_pd.shape)
print("Catchment dataset shape:", catchments_pd.shape)

print("\nGPS columns:")
print(gps_pd.columns.tolist())

print("\nStore columns:")
print(stores_pd.columns.tolist())

print("\nCatchment columns:")
print(catchments_pd.columns.tolist())

GPS dataset shape: (140000, 8)
Store dataset shape: (20, 6)
Catchment dataset shape: (20, 9)

GPS columns:
['DeviceID', 'latitude', 'longitude', 'timestamp', 'activity_period', 'activity_zone', 'day_name', 'day_type']

Store columns:
['StoreID', 'StoreName', 'latitude', 'longitude', 'Category', 'StoreSize_sqft']

Catchment columns:
['StoreID', 'StoreName', 'latitude', 'longitude', 'Category', 'StoreSize_sqft', 'catchment_wkt', 'buffer_radius_m', 'area_sq_meters']


In [5]:
!pip install -q pyspark==3.5.3 apache-sedona==1.7.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


In [6]:
import os

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from sedona.spark import SedonaContext

spark = (
    SparkSession.builder
    .appName("GeoPulse-Day8-Spatial-Join")
    .config(
        "spark.jars.packages",
        "org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.7.1,"
        "org.datasyslab:geotools-wrapper:1.7.1-28.5"
    )
    .getOrCreate()
)

sedona = SedonaContext.create(spark)

print("Spark version:", spark.version)
print("Apache Sedona initialized successfully!")

Spark version: 3.5.3
Apache Sedona initialized successfully!


In [7]:
gps_spark_df = spark.createDataFrame(gps_pd)

print("GPS records loaded:", gps_spark_df.count())

gps_spark_df.printSchema()

gps_spark_df.show(5, truncate=False)

GPS records loaded: 140000
root
 |-- DeviceID: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- activity_period: string (nullable = true)
 |-- activity_zone: string (nullable = true)
 |-- day_name: string (nullable = true)
 |-- day_type: string (nullable = true)

+------------+------------------+-----------------+-------------------+---------------+-------------+---------+--------+
|DeviceID    |latitude          |longitude        |timestamp          |activity_period|activity_zone|day_name |day_type|
+------------+------------------+-----------------+-------------------+---------------+-------------+---------+--------+
|DEVICE_00001|18.49987772411253 |73.8517777565856 |2026-11-18 06:50:06|morning        |residential  |Wednesday|Weekday |
|DEVICE_00001|18.501044317975023|73.84020164938059|2026-11-18 07:03:53|morning        |residential  |Wednesday|Weekday |
|DEVICE_00001|18.56045880388

In [8]:
catchments_spark_df = spark.createDataFrame(catchments_pd)

print("Catchment polygons loaded:", catchments_spark_df.count())

catchments_spark_df.select(
    "StoreID",
    "StoreName",
    "buffer_radius_m",
    "area_sq_meters",
    "catchment_wkt"
).show(3, truncate=False)

Catchment polygons loaded: 20
+------------+-----------------------+---------------+-----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
from pyspark.sql import functions as F

catchments_geometry_df = (
    catchments_spark_df
    .withColumn(
        "catchment_geometry",
        F.expr("ST_GeomFromWKT(catchment_wkt)")
    )
)

catchments_geometry_df.select(
    "StoreID",
    "StoreName",
    F.expr("ST_GeometryType(catchment_geometry)").alias("geometry_type"),
    F.expr("ST_IsValid(catchment_geometry)").alias("is_valid")
).show(20, truncate=False)

+------------+------------------------+-------------+--------+
|StoreID     |StoreName               |geometry_type|is_valid|
+------------+------------------------+-------------+--------+
|GP_STORE_001|GeoPulse Retail Store 1 |ST_Polygon   |true    |
|GP_STORE_002|GeoPulse Retail Store 2 |ST_Polygon   |true    |
|GP_STORE_003|GeoPulse Retail Store 3 |ST_Polygon   |true    |
|GP_STORE_004|GeoPulse Retail Store 4 |ST_Polygon   |true    |
|GP_STORE_005|GeoPulse Retail Store 5 |ST_Polygon   |true    |
|GP_STORE_006|GeoPulse Retail Store 6 |ST_Polygon   |true    |
|GP_STORE_007|GeoPulse Retail Store 7 |ST_Polygon   |true    |
|GP_STORE_008|GeoPulse Retail Store 8 |ST_Polygon   |true    |
|GP_STORE_009|GeoPulse Retail Store 9 |ST_Polygon   |true    |
|GP_STORE_010|GeoPulse Retail Store 10|ST_Polygon   |true    |
|GP_STORE_011|GeoPulse Retail Store 11|ST_Polygon   |true    |
|GP_STORE_012|GeoPulse Retail Store 12|ST_Polygon   |true    |
|GP_STORE_013|GeoPulse Retail Store 13|ST_Polygon   |tr

In [10]:
gps_points_df = (
    gps_spark_df
    .withColumn(
        "gps_geometry",
        F.expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
    )
    .withColumn(
        "projected_geometry",
        F.expr("ST_Transform(gps_geometry, 'epsg:4326', 'epsg:32643')")
    )
)

print("GPS points created:", gps_points_df.count())

gps_points_df.select(
    "DeviceID",
    "latitude",
    "longitude",
    F.expr("ST_GeometryType(projected_geometry)").alias("geometry_type")
).show(5, truncate=False)

GPS points created: 140000
+------------+------------------+-----------------+-------------+
|DeviceID    |latitude          |longitude        |geometry_type|
+------------+------------------+-----------------+-------------+
|DEVICE_00001|18.49987772411253 |73.8517777565856 |ST_Point     |
|DEVICE_00001|18.501044317975023|73.84020164938059|ST_Point     |
|DEVICE_00001|18.560458803882675|73.900062155427  |ST_Point     |
|DEVICE_00001|18.558067635247177|73.90420614879037|ST_Point     |
|DEVICE_00001|18.564864817668777|73.91397797731948|ST_Point     |
+------------+------------------+-----------------+-------------+
only showing top 5 rows



In [11]:
gps_store_join_df = (
    gps_points_df.alias("gps")
    .join(
        catchments_geometry_df.alias("store"),
        F.expr(
            "ST_Contains(store.catchment_geometry, gps.projected_geometry)"
        ),
        "inner"
    )
    .select(
        F.col("gps.DeviceID").alias("DeviceID"),
        F.col("gps.latitude").alias("GPS_Latitude"),
        F.col("gps.longitude").alias("GPS_Longitude"),
        F.col("gps.timestamp").alias("timestamp"),
        F.col("store.StoreID").alias("StoreID"),
        F.col("store.StoreName").alias("StoreName"),
        F.col("store.Category").alias("Category")
    )
)

print("GPS-to-store spatial join completed!")

print("Matching GPS records:", gps_store_join_df.count())

gps_store_join_df.show(10, truncate=False)

GPS-to-store spatial join completed!
Matching GPS records: 26
+------------+------------------+-----------------+-------------------+------------+-----------------------+-----------+
|DeviceID    |GPS_Latitude      |GPS_Longitude    |timestamp          |StoreID     |StoreName              |Category   |
+------------+------------------+-----------------+-------------------+------------+-----------------------+-----------+
|DEVICE_00190|18.579148910856706|73.90118512126296|2026-11-23 14:18:56|GP_STORE_002|GeoPulse Retail Store 2|Electronics|
|DEVICE_00702|18.584175603841963|73.90310701658188|2026-11-23 15:40:38|GP_STORE_002|GeoPulse Retail Store 2|Electronics|
|DEVICE_00890|18.548574147691838|73.92288712668409|2026-11-20 18:19:03|GP_STORE_005|GeoPulse Retail Store 5|Lifestyle  |
|DEVICE_00379|18.54880772097283 |73.92291796770044|2026-11-24 19:19:26|GP_STORE_005|GeoPulse Retail Store 5|Lifestyle  |
|DEVICE_00345|18.54951774342487 |73.92400501444894|2026-11-24 10:43:20|GP_STORE_005|GeoPuls

In [12]:
print("GPS projected coordinate range:")

gps_points_df.select(
    F.min(F.expr("ST_X(projected_geometry)")).alias("min_x"),
    F.max(F.expr("ST_X(projected_geometry)")).alias("max_x"),
    F.min(F.expr("ST_Y(projected_geometry)")).alias("min_y"),
    F.max(F.expr("ST_Y(projected_geometry)")).alias("max_y")
).show()

print("Catchment coordinate range:")

catchments_geometry_df.select(
    F.min(F.expr("ST_XMin(catchment_geometry)")).alias("min_x"),
    F.max(F.expr("ST_XMax(catchment_geometry)")).alias("max_x"),
    F.min(F.expr("ST_YMin(catchment_geometry)")).alias("min_y"),
    F.max(F.expr("ST_YMax(catchment_geometry)")).alias("max_y")
).show()

GPS projected coordinate range:
+------------------+------------------+------------------+-----------------+
|             min_x|             max_x|             min_y|            max_y|
+------------------+------------------+------------------+-----------------+
|376509.29053129954|386968.98585353146|2043521.4611520495|2055171.529211958|
+------------------+------------------+------------------+-----------------+

Catchment coordinate range:
+------------------+------------------+------------------+------------------+
|             min_x|             max_x|             min_y|             max_y|
+------------------+------------------+------------------+------------------+
|368803.87976538256|394082.82602854416|2034805.7165327887|2061973.9133438154|
+------------------+------------------+------------------+------------------+



In [13]:
from pyspark.sql import functions as F

store_match_counts = (
    gps_store_join_df
    .groupBy("StoreID", "StoreName")
    .agg(
        F.count("*").alias("matching_gps_records"),
        F.countDistinct("DeviceID").alias("unique_devices")
    )
    .orderBy("StoreID")
)

store_match_counts.show(25, truncate=False)

print("Stores with at least one matching GPS record:",
      store_match_counts.count())

+------------+-----------------------+--------------------+--------------+
|StoreID     |StoreName              |matching_gps_records|unique_devices|
+------------+-----------------------+--------------------+--------------+
|GP_STORE_002|GeoPulse Retail Store 2|2                   |2             |
|GP_STORE_005|GeoPulse Retail Store 5|24                  |23            |
+------------+-----------------------+--------------------+--------------+

Stores with at least one matching GPS record: 2


In [14]:
print("Store locations:")
stores_pd[["StoreID", "StoreName", "latitude", "longitude"]].sort_values("StoreID")

Store locations:


,StoreID,StoreName,latitude,longitude
0,GP_STORE_001,GeoPulse Retail Store 1,18.493635,73.987679
1,GP_STORE_002,GeoPulse Retail Store 2,18.582998,73.899665
2,GP_STORE_003,GeoPulse Retail Store 3,18.439005,73.788999
3,GP_STORE_004,GeoPulse Retail Store 4,18.414521,73.966544
4,GP_STORE_005,GeoPulse Retail Store 5,18.550279,73.927018
5,GP_STORE_006,GeoPulse Retail Store 6,18.405146,73.992477
6,GP_STORE_007,GeoPulse Retail Store 7,18.608111,73.803085
7,GP_STORE_008,GeoPulse Retail Store 8,18.445456,73.795851
8,GP_STORE_009,GeoPulse Retail Store 9,18.476061,73.881189
9,GP_STORE_010,GeoPulse Retail Store 10,18.507986,73.822807


In [15]:
print("GPS coordinate range:")
print(gps_pd[["latitude", "longitude"]].agg(["min", "max"]))

print("\nGPS sample:")
display(gps_pd[["latitude", "longitude", "activity_zone"]].head(10))

GPS coordinate range:
      latitude  longitude
min  18.478523  73.830206
max  18.584176  73.928877

GPS sample:


,latitude,longitude,activity_zone
0,18.499878,73.851778,residential
1,18.501044,73.840202,residential
2,18.560459,73.900062,office
3,18.558068,73.904206,office
4,18.564865,73.913978,office
5,18.562289,73.901579,office
6,18.567328,73.908871,office
7,18.489108,73.844781,residential
8,18.531620,73.868075,food
9,18.559159,73.915824,office


In [16]:
display(
    stores_pd[
        ["StoreID", "StoreName", "latitude", "longitude"]
    ].sort_values("StoreID").to_string(index=False)
)

'     StoreID                StoreName  latitude  longitude\nGP_STORE_001  GeoPulse Retail Store 1 18.493635  73.987679\nGP_STORE_002  GeoPulse Retail Store 2 18.582998  73.899665\nGP_STORE_003  GeoPulse Retail Store 3 18.439005  73.788999\nGP_STORE_004  GeoPulse Retail Store 4 18.414521  73.966544\nGP_STORE_005  GeoPulse Retail Store 5 18.550279  73.927018\nGP_STORE_006  GeoPulse Retail Store 6 18.405146  73.992477\nGP_STORE_007  GeoPulse Retail Store 7 18.608111  73.803085\nGP_STORE_008  GeoPulse Retail Store 8 18.445456  73.795851\nGP_STORE_009  GeoPulse Retail Store 9 18.476061  73.881189\nGP_STORE_010 GeoPulse Retail Store 10 18.507986  73.822807\nGP_STORE_011 GeoPulse Retail Store 11 18.552963  73.784873\nGP_STORE_012 GeoPulse Retail Store 12 18.473036  73.841590\nGP_STORE_013 GeoPulse Retail Store 13 18.514017  73.946294\nGP_STORE_014 GeoPulse Retail Store 14 18.449918  73.878559\nGP_STORE_015 GeoPulse Retail Store 15 18.548104  73.761613\nGP_STORE_016 GeoPulse Retail Store 16 1

In [17]:
from pyspark.sql import functions as F

store_points_df = (
    spark.createDataFrame(stores_pd)
    .withColumn(
        "store_geometry",
        F.expr(
            "ST_Transform("
            "ST_SetSRID(ST_Point(longitude, latitude), 4326), "
            "'epsg:4326', 'epsg:32643')"
        )
    )
)

nearest_distances_df = (
    store_points_df.alias("s")
    .crossJoin(gps_points_df.select("projected_geometry").alias("g"))
    .withColumn(
        "distance_m",
        F.expr("ST_Distance(s.store_geometry, g.projected_geometry)")
    )
    .groupBy("StoreID", "StoreName")
    .agg(F.min("distance_m").alias("nearest_gps_distance_m"))
    .orderBy("StoreID")
)

nearest_distances_df.show(20, truncate=False)

+------------+------------------------+----------------------+
|StoreID     |StoreName               |nearest_gps_distance_m|
+------------+------------------------+----------------------+
|GP_STORE_001|GeoPulse Retail Store 1 |9016.304332501637     |
|GP_STORE_002|GeoPulse Retail Store 2 |385.9127379037385     |
|GP_STORE_003|GeoPulse Retail Store 3 |6975.575264196149     |
|GP_STORE_004|GeoPulse Retail Store 4 |12542.738296070764    |
|GP_STORE_005|GeoPulse Retail Store 5 |226.24293053504525    |
|GP_STORE_006|GeoPulse Retail Store 6 |15115.497563221235    |
|GP_STORE_007|GeoPulse Retail Store 7 |8953.716726565488     |
|GP_STORE_008|GeoPulse Retail Store 8 |5964.9944039081465    |
|GP_STORE_009|GeoPulse Retail Store 9 |2190.8958511239803    |
|GP_STORE_010|GeoPulse Retail Store 10|904.4727676927558     |
|GP_STORE_011|GeoPulse Retail Store 11|6963.545774485903     |
|GP_STORE_012|GeoPulse Retail Store 12|618.8400090331353     |
|GP_STORE_013|GeoPulse Retail Store 13|4401.28585158040

In [18]:
print("Store latitude range:")
print(stores_pd["latitude"].agg(["min", "max"]))

print("\nStore longitude range:")
print(stores_pd["longitude"].agg(["min", "max"]))

print("\nStore count:", stores_pd["StoreID"].nunique())


Store latitude range:
min    18.405146
max    18.641408
Name: latitude, dtype: float64

Store longitude range:
min    73.761613
max    73.992477
Name: longitude, dtype: float64

Store count: 20


In [19]:
nearby_gps_counts = (
    store_points_df.alias("s")
    .crossJoin(
        gps_points_df.select("projected_geometry").alias("g")
    )
    .withColumn(
        "distance_m",
        F.expr("ST_Distance(s.store_geometry, g.projected_geometry)")
    )
    .filter(F.col("distance_m") <= 1000)
    .groupBy("StoreID")
    .agg(F.count("*").alias("gps_points_within_1km"))
    .orderBy("StoreID")
)

nearby_gps_counts.show(20, truncate=False)

+------------+---------------------+
|StoreID     |gps_points_within_1km|
+------------+---------------------+
|GP_STORE_002|18                   |
|GP_STORE_005|442                  |
|GP_STORE_010|2                    |
|GP_STORE_012|1                    |
+------------+---------------------+



In [20]:
print("Activity periods:")
print(gps_pd["activity_period"].value_counts())

print("\nActivity zones:")
print(gps_pd["activity_zone"].value_counts())

print("\nDay types:")
print(gps_pd["day_type"].value_counts())

Activity periods:
activity_period
midday     47651
evening    38336
morning    29222
night      24791
Name: count, dtype: int64

Activity zones:
activity_zone
shopping       36515
residential    34903
office         34576
food           18113
leisure        15893
Name: count, dtype: int64

Day types:
day_type
Weekday    100000
Weekend     40000
Name: count, dtype: int64


In [21]:
import numpy as np
import pandas as pd

np.random.seed(42)

supplementary_records = []

activity_periods = ["morning", "midday", "evening", "night"]
activity_zones = [
    "shopping", "residential", "office", "food", "leisure"
]

for _, store in stores_pd.iterrows():

    for i in range(500):

        # Generate a nearby location around the store
        latitude = store["latitude"] + np.random.normal(0, 0.0025)
        longitude = store["longitude"] + np.random.normal(0, 0.0025)

        supplementary_records.append({
            "DeviceID": f"SUP_{store['StoreID']}_{i:04d}",
            "latitude": latitude,
            "longitude": longitude,
            "timestamp": pd.Timestamp("2026-11-20")
                + pd.Timedelta(minutes=int(np.random.randint(0, 10080))),
            "activity_period": np.random.choice(activity_periods),
            "activity_zone": np.random.choice(activity_zones),
            "day_name": np.random.choice([
                "Monday", "Tuesday", "Wednesday",
                "Thursday", "Friday", "Saturday", "Sunday"
            ]),
            "day_type": np.random.choice(["Weekday", "Weekend"])
        })

supplementary_gps_pd = pd.DataFrame(supplementary_records)

print("Supplementary GPS records:", len(supplementary_gps_pd))
print("Unique supplementary devices:",
      supplementary_gps_pd["DeviceID"].nunique())

display(supplementary_gps_pd.head())

Supplementary GPS records: 10000
Unique supplementary devices: 10000


,DeviceID,latitude,longitude,timestamp,activity_period,activity_zone,day_name,day_type
0,SUP_GP_STORE_001_0000,18.494877,73.987333,2026-11-23 14:31:00,morning,leisure,Sunday,Weekend
1,SUP_GP_STORE_001_0001,18.494333,73.990205,2026-11-25 18:42:00,midday,leisure,Tuesday,Weekend
2,SUP_GP_STORE_001_0002,18.492461,73.989035,2026-11-23 16:31:00,night,leisure,Monday,Weekend
3,SUP_GP_STORE_001_0003,18.493691,73.986609,2026-11-21 18:38:00,midday,food,Thursday,Weekend
4,SUP_GP_STORE_001_0004,18.493007,73.987269,2026-11-22 02:05:00,evening,food,Sunday,Weekend


In [22]:
import numpy as np
import pandas as pd

np.random.seed(42)

supplementary_records = []

for _, store in stores_pd.iterrows():
    for i in range(500):

        # Generate timestamp within Nov 18–24, 2026
        timestamp = (
            pd.Timestamp("2026-11-18")
            + pd.Timedelta(seconds=int(np.random.randint(0, 7 * 24 * 60 * 60)))
        )

        hour = timestamp.hour

        if 5 <= hour < 11:
            activity_period = "morning"
        elif 11 <= hour < 16:
            activity_period = "midday"
        elif 16 <= hour < 21:
            activity_period = "evening"
        else:
            activity_period = "night"

        supplementary_records.append({
            "DeviceID": f"SUP_{store['StoreID']}_{i:04d}",
            "latitude": store["latitude"] + np.random.normal(0, 0.0025),
            "longitude": store["longitude"] + np.random.normal(0, 0.0025),
            "timestamp": timestamp,
            "activity_period": activity_period,
            "activity_zone": np.random.choice([
                "shopping", "residential", "office", "food", "leisure"
            ]),
            "day_name": timestamp.day_name(),
            "day_type": "Weekend" if timestamp.dayofweek >= 5 else "Weekday"
        })

supplementary_gps_pd = pd.DataFrame(supplementary_records)

print("Records:", len(supplementary_gps_pd))
print("Unique devices:", supplementary_gps_pd["DeviceID"].nunique())
print("\nDay type counts:")
print(supplementary_gps_pd["day_type"].value_counts())
print("\nActivity period counts:")
print(supplementary_gps_pd["activity_period"].value_counts())

display(supplementary_gps_pd.head())

Records: 10000
Unique devices: 10000

Day type counts:
day_type
Weekday    7073
Weekend    2927
Name: count, dtype: int64

Activity period counts:
activity_period
night      3274
morning    2543
evening    2095
midday     2088
Name: count, dtype: int64


,DeviceID,latitude,longitude,timestamp,activity_period,activity_zone,day_name,day_type
0,SUP_GP_STORE_001_0000,18.492259,73.988967,2026-11-19 09:52:38,morning,leisure,Thursday,Weekday
1,SUP_GP_STORE_001_0001,18.491343,73.987368,2026-11-18 15:14:46,midday,office,Wednesday,Weekday
2,SUP_GP_STORE_001_0002,18.494616,73.985356,2026-11-20 00:40:03,night,leisure,Friday,Weekday
3,SUP_GP_STORE_001_0003,18.492461,73.989035,2026-11-21 17:24:39,evening,food,Saturday,Weekend
4,SUP_GP_STORE_001_0004,18.497291,73.991525,2026-11-20 17:29:56,evening,leisure,Friday,Weekday


In [24]:
combined_gps_pd["timestamp"] = pd.to_datetime(
    combined_gps_pd["timestamp"],
    errors="coerce"
)

print("Combined GPS records:", len(combined_gps_pd))
print("Unique devices:", combined_gps_pd["DeviceID"].nunique())
print("Invalid timestamps:", combined_gps_pd["timestamp"].isna().sum())

print("\nDate range:")
print(combined_gps_pd["timestamp"].min())
print(combined_gps_pd["timestamp"].max())

Combined GPS records: 150000
Unique devices: 11000
Invalid timestamps: 0

Date range:
2026-11-18 00:03:29
2026-11-24 23:59:58


In [26]:
text_columns = [
    "DeviceID",
    "activity_period",
    "activity_zone",
    "day_name",
    "day_type"
]

for col in text_columns:
    combined_gps_pd[col] = combined_gps_pd[col].astype(str)

combined_gps_pd["timestamp"] = pd.to_datetime(
    combined_gps_pd["timestamp"]
)

print(combined_gps_pd.dtypes)
print("\nNull values:")
print(combined_gps_pd.isnull().sum())

DeviceID                   object
latitude                  float64
longitude                 float64
timestamp          datetime64[ns]
activity_period            object
activity_zone              object
day_name                   object
day_type                   object
dtype: object

Null values:
DeviceID           0
latitude           0
longitude          0
timestamp          0
activity_period    0
activity_zone      0
day_name           0
day_type           0
dtype: int64


In [28]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, TimestampType
)

gps_schema = StructType([
    StructField("DeviceID", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("activity_period", StringType(), False),
    StructField("activity_zone", StringType(), False),
    StructField("day_name", StringType(), False),
    StructField("day_type", StringType(), False)
])

gps_rows = [
    (
        str(row.DeviceID),
        float(row.latitude),
        float(row.longitude),
        row.timestamp.to_pydatetime(),
        str(row.activity_period),
        str(row.activity_zone),
        str(row.day_name),
        str(row.day_type)
    )
    for row in combined_gps_pd.itertuples(index=False)
]

gps_combined_spark_df = spark.createDataFrame(
    gps_rows,
    schema=gps_schema
)

print("Spark GPS record count:", gps_combined_spark_df.count())
gps_combined_spark_df.printSchema()

Spark GPS record count: 150000
root
 |-- DeviceID: string (nullable = false)
 |-- latitude: double (nullable = false)
 |-- longitude: double (nullable = false)
 |-- timestamp: timestamp (nullable = false)
 |-- activity_period: string (nullable = false)
 |-- activity_zone: string (nullable = false)
 |-- day_name: string (nullable = false)
 |-- day_type: string (nullable = false)



In [29]:
from pyspark.sql import functions as F

gps_combined_points_df = (
    gps_combined_spark_df
    .withColumn(
        "gps_geometry",
        F.expr(
            "ST_SetSRID(ST_Point(longitude, latitude), 4326)"
        )
    )
    .withColumn(
        "projected_geometry",
        F.expr(
            "ST_Transform(gps_geometry, 'epsg:4326', 'epsg:32643')"
        )
    )
)

print("GPS records with geometry:",
      gps_combined_points_df.count())

gps_combined_points_df.select(
    "DeviceID",
    "latitude",
    "longitude",
    F.expr("ST_GeometryType(projected_geometry)").alias("geometry_type")
).show(5, truncate=False)

GPS records with geometry: 150000
+------------+------------------+-----------------+-------------+
|DeviceID    |latitude          |longitude        |geometry_type|
+------------+------------------+-----------------+-------------+
|DEVICE_00001|18.49987772411253 |73.8517777565856 |ST_Point     |
|DEVICE_00001|18.501044317975023|73.84020164938059|ST_Point     |
|DEVICE_00001|18.560458803882675|73.900062155427  |ST_Point     |
|DEVICE_00001|18.558067635247177|73.90420614879037|ST_Point     |
|DEVICE_00001|18.564864817668777|73.91397797731948|ST_Point     |
+------------+------------------+-----------------+-------------+
only showing top 5 rows



In [30]:
gps_store_join_df = (
    gps_combined_points_df.alias("gps")
    .join(
        catchments_geometry_df.alias("store"),
        F.expr(
            "ST_Contains("
            "store.catchment_geometry, "
            "gps.projected_geometry)"
        ),
        "inner"
    )
    .select(
        F.col("gps.DeviceID").alias("DeviceID"),
        F.col("gps.latitude").alias("GPS_Latitude"),
        F.col("gps.longitude").alias("GPS_Longitude"),
        F.col("gps.timestamp").alias("timestamp"),
        F.col("gps.activity_period").alias("activity_period"),
        F.col("gps.activity_zone").alias("activity_zone"),
        F.col("gps.day_type").alias("day_type"),
        F.col("store.StoreID").alias("StoreID"),
        F.col("store.StoreName").alias("StoreName"),
        F.col("store.Category").alias("Category")
    )
)

print("Total GPS-to-store matches:", gps_store_join_df.count())

gps_store_join_df.show(10, truncate=False)

Total GPS-to-store matches: 8250
+---------------------+------------------+-----------------+-------------------+---------------+-------------+--------+------------+-----------------------+--------+
|DeviceID             |GPS_Latitude      |GPS_Longitude    |timestamp          |activity_period|activity_zone|day_type|StoreID     |StoreName              |Category|
+---------------------+------------------+-----------------+-------------------+---------------+-------------+--------+------------+-----------------------+--------+
|SUP_GP_STORE_003_0124|18.43475591653891 |73.79020126331187|2026-11-21 22:53:29|night          |residential  |Weekend |GP_STORE_003|GeoPulse Retail Store 3|Fashion |
|SUP_GP_STORE_003_0352|18.434841387413915|73.78826982680675|2026-11-24 18:44:02|evening        |shopping     |Weekday |GP_STORE_003|GeoPulse Retail Store 3|Fashion |
|SUP_GP_STORE_003_0249|18.4349279556583  |73.78817142974809|2026-11-19 07:59:00|morning        |office       |Weekday |GP_STORE_003|GeoPu

In [31]:
store_reach_df = (
    gps_store_join_df
    .groupBy("StoreID", "StoreName", "Category")
    .agg(
        F.count("*").alias("gps_ping_count"),
        F.countDistinct("DeviceID").alias("unique_simulated_devices")
    )
    .orderBy("StoreID")
)

store_reach_df.show(20, truncate=False)

print(
    "Stores with GPS activity:",
    store_reach_df.count()
)

+------------+------------------------+-----------+--------------+------------------------+
|StoreID     |StoreName               |Category   |gps_ping_count|unique_simulated_devices|
+------------+------------------------+-----------+--------------+------------------------+
|GP_STORE_001|GeoPulse Retail Store 1 |Supermarket|399           |399                     |
|GP_STORE_002|GeoPulse Retail Store 2 |Electronics|420           |420                     |
|GP_STORE_003|GeoPulse Retail Store 3 |Fashion    |418           |418                     |
|GP_STORE_004|GeoPulse Retail Store 4 |Supermarket|403           |403                     |
|GP_STORE_005|GeoPulse Retail Store 5 |Lifestyle  |421           |420                     |
|GP_STORE_006|GeoPulse Retail Store 6 |Lifestyle  |406           |406                     |
|GP_STORE_007|GeoPulse Retail Store 7 |Supermarket|410           |410                     |
|GP_STORE_008|GeoPulse Retail Store 8 |Supermarket|417           |417           

In [32]:
device_store_counts = (
    gps_store_join_df
    .groupBy("DeviceID")
    .agg(
        F.countDistinct("StoreID").alias("stores_visited")
    )
)

multi_store_devices_df = (
    device_store_counts
    .filter(F.col("stores_visited") > 1)
)

print(
    "Devices appearing in multiple store catchments:",
    multi_store_devices_df.count()
)

print("\nDistribution of store visits:")
multi_store_devices_df.groupBy("stores_visited").count().orderBy(
    "stores_visited"
).show()

Devices appearing in multiple store catchments: 37

Distribution of store visits:
+--------------+-----+
|stores_visited|count|
+--------------+-----+
|             2|   37|
+--------------+-----+



In [33]:
multi_store_device_ids = (
    multi_store_devices_df
    .select("DeviceID")
)

store_overlap_df = (
    gps_store_join_df
    .join(
        multi_store_device_ids,
        on="DeviceID",
        how="inner"
    )
    .groupBy("StoreID", "StoreName")
    .agg(
        F.countDistinct("DeviceID").alias(
            "devices_in_multiple_catchments"
        )
    )
    .orderBy("StoreID")
)

store_overlap_df.show(20, truncate=False)

+------------+------------------------+------------------------------+
|StoreID     |StoreName               |devices_in_multiple_catchments|
+------------+------------------------+------------------------------+
|GP_STORE_011|GeoPulse Retail Store 11|37                            |
|GP_STORE_016|GeoPulse Retail Store 16|37                            |
+------------+------------------------+------------------------------+



In [34]:
store_mobility_df = (
    catchments_spark_df
    .select(
        "StoreID",
        "StoreName",
        "Category",
        "buffer_radius_m",
        "area_sq_meters"
    )
    .join(
        store_reach_df,
        ["StoreID", "StoreName", "Category"],
        "left"
    )
    .join(
        store_overlap_df.select(
            "StoreID",
            "devices_in_multiple_catchments"
        ),
        "StoreID",
        "left"
    )
    .fillna({
        "gps_ping_count": 0,
        "unique_simulated_devices": 0,
        "devices_in_multiple_catchments": 0
    })
    .withColumn(
        "multi_store_overlap_rate_pct",
        F.when(
            F.col("unique_simulated_devices") > 0,
            F.round(
                F.col("devices_in_multiple_catchments")
                / F.col("unique_simulated_devices") * 100,
                2
            )
        ).otherwise(0)
    )
    .orderBy("StoreID")
)

store_mobility_df.show(20, truncate=False)
print("Total stores in analysis:", store_mobility_df.count())

+------------+------------------------+-----------+---------------+-----------------+--------------+------------------------+------------------------------+----------------------------+
|StoreID     |StoreName               |Category   |buffer_radius_m|area_sq_meters   |gps_ping_count|unique_simulated_devices|devices_in_multiple_catchments|multi_store_overlap_rate_pct|
+------------+------------------------+-----------+---------------+-----------------+--------------+------------------------+------------------------------+----------------------------+
|GP_STORE_001|GeoPulse Retail Store 1 |Supermarket|500            |780361.2880645435|399           |399                     |0                             |0.0                         |
|GP_STORE_002|GeoPulse Retail Store 2 |Electronics|500            |780361.2880645435|420           |420                     |0                             |0.0                         |
|GP_STORE_003|GeoPulse Retail Store 3 |Fashion    |500            |780

In [35]:
output_path = (
    repo_path / "data/synthetic/store_mobility_analysis.csv"
)

store_mobility_pd = store_mobility_df.toPandas()

store_mobility_pd.to_csv(output_path, index=False)

print("Export successful!")
print("File:", output_path)
print("Rows:", len(store_mobility_pd))
print("Columns:", len(store_mobility_pd.columns))
print("File size (KB):", round(output_path.stat().st_size / 1024, 2))


Export successful!
File: /content/gdrive/MyDrive/GeoPulse-GitHub-Repo/data/synthetic/store_mobility_analysis.csv
Rows: 20
Columns: 9
File size (KB): 1.8



## Day 8 – GPS-to-Store Spatial Join

### Work completed
- Loaded 140,000 original synthetic GPS records.
- Generated 10,000 supplementary synthetic GPS observations to improve spatial coverage around all 20 stores.
- Combined dataset: 150,000 GPS observations.
- Converted GPS coordinates to point geometries.
- Used Apache Sedona to join GPS points with 500-meter store catchment polygons.
- Identified 8,250 GPS-to-store spatial matches.
- Calculated GPS ping counts and distinct device IDs per store.
- Identified 37 device IDs appearing in both Store 11 and Store 16 catchments.
- Exported the final 20-store analysis to `data/synthetic/store_mobility_analysis.csv`.

### Key findings
- All 20 stores have at least one matching GPS observation.
- Store 11 and Store 16 share 37 simulated device IDs.
- Their calculated overlap rates are 8.13% and 8.31%, respectively.

### Methodology and limitations
- All GPS data and store locations are synthetic.
- Supplementary records were generated to improve spatial coverage and validate the spatial join.
- Each supplementary record has its own device ID and represents one synthetic observation, not a verified shopper or repeat visitor.
- Results describe simulated spatial reach and overlap, not actual customer behaviour or proven retail cannibalization.
- Catchments are fixed 500-meter buffers, not customer-derived boundaries.

### Day 8 deliverable
`data/synthetic/store_mobility_analysis.csv`
